In [1]:
import sys
sys.path.append("..")

from src.scraper import (
    criar_driver,
    extrair_dados_luta,
    extrair_urls_evento,
    extrair_evento,
    extrair_varios_eventos,
    extrair_eventos_multiplas_paginas,
)
from src.limpeza import (
    separar_landed_attempted,
    limpar_percentual,
    tempo_para_segundos,
)
from src.config import COLUNAS_OF, COLUNAS_PCT

import pandas as pd
import time

colunas_of = COLUNAS_OF
colunas_pct = COLUNAS_PCT

In [43]:
import requests
from bs4 import BeautifulSoup

In [44]:
import os

In [45]:
url = "http://ufcstats.com/fight-details/568ec6af4008355a"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
print(resp.status_code)

html = resp.text

200


In [46]:
soup = BeautifulSoup(html, "html.parser")

In [47]:
lutadores = soup.find_all("div", class_="b-fight-details__person")

for pessoa in lutadores:
    nome = pessoa.find("a", class_="b-link b-fight-details__person-link").text.strip()
    resultado = pessoa.find("i", class_="b-fight-details__person-status").text.strip()
    print(nome, resultado)

In [48]:
print(len(lutadores))

0


In [49]:
print(html[:500])

<!doctype html><html><head><meta charset="utf-8">
<title>Loading…</title><meta name="robots" content="noindex">
<style>body{font-family:sans-serif;color:#666;text-align:center;margin-top:25vh}</style>
</head><body>
<p>Checking your browser…</p>
<noscript>This site requires JavaScript.</noscript>
<script>
(function(){
var K=[0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,0x3956c25b,0x59f111f1,0x923f82a4,0xab1c5ed5,
0xd807aa98,0x12835b01,0x243185be,0x550c7dc3,0x72be5d74,0x80deb1fe,0x9bdc06a7,0xc19bf1


In [50]:
driver = criar_driver()

In [51]:
import time

url = "http://ufcstats.com/fight-details/568ec6af4008355a"
driver.get(url)

time.sleep(5)  # espera a verificação/JS terminar de carregar

html = driver.page_source
print(html[:500])

KeyboardInterrupt: 

In [ ]:
soup = BeautifulSoup(html, "html.parser")
lutadores = soup.find_all("div", class_="b-fight-details__person")
print(len(lutadores))

2


In [ ]:
for pessoa in lutadores:
    nome = pessoa.find("a", class_="b-link b-fight-details__person-link").text.strip()
    resultado = pessoa.find("i", class_="b-fight-details__person-status").text.strip()
    print(nome, resultado)

Joshua Van W
Alexandre Pantoja L


In [ ]:
info_bloco = soup.find("p", class_="b-fight-details__text")
itens = info_bloco.find_all("i", class_=["b-fight-details__text-item_first", "b-fight-details__text-item"])

for item in itens:
    label = item.find("i", class_="b-fight-details__label").text.strip()
    valor = item.get_text().replace(label, "").strip()
    print(label, "->", valor)

Method: -> Decision - Unanimous
Round: -> 5
Time: -> 5:00
Time format: -> 5 Rnd (5-5-5-5-5)
Referee: -> Herb Dean


In [ ]:
from io import StringIO
import pandas as pd

tabelas = pd.read_html(StringIO(html))
print(len(tabelas))

4


In [ ]:
for i, tabela in enumerate(tabelas):
    print(f"--- Tabela {i} ---")
    print(tabela.head())
    print()

--- Tabela 0 ---
                        Fighter    KD               Sig. str. Sig. str. %  \
0  Joshua Van Alexandre Pantoja  1  0  181 of 305  147 of 284    59%  51%   

               Total str.               Td      Td % Sub. att  Rev.  \
0  259 of 392  159 of 300  0 of 0  5 of 18  ---  27%     1  0  1  0   

         Ctrl  
0  2:13  9:11  

--- Tabela 1 ---
                        Fighter      KD           Sig. str. Sig. str. %  \
                        Round 1 Round 1             Round 1     Round 1   
                        Round 2 Round 2             Round 2     Round 2   
                        Round 3 Round 3             Round 3     Round 3   
                        Round 4 Round 4             Round 4     Round 4   
                        Round 5 Round 5             Round 5     Round 5   
0  Joshua Van Alexandre Pantoja    0  0  13 of 27  27 of 45    48%  60%   
1  Joshua Van Alexandre Pantoja    0  0  37 of 64  32 of 76    57%  42%   
2  Joshua Van Alexandre Pantoja    

In [ ]:
def extrair_tabela_por_lutador(tabela_html):
    # Cabeçalhos das colunas
    headers = [th.get_text(strip=True) for th in tabela_html.find_all("th")]
    
    # Todas as linhas de dados (ignorando a linha de cabeçalho)
    linhas = tabela_html.find_all("tr")[1:]  # [1:] pula o <tr> do cabeçalho
    
    dados_lutador_1 = []
    dados_lutador_2 = []
    
    for linha in linhas:
        celulas = linha.find_all("td", class_="b-fight-details__table-col")
        valores_1 = []
        valores_2 = []
        for celula in celulas:
            ps = celula.find_all("p", class_="b-fight-details__table-text")
            valores_1.append(ps[0].get_text(strip=True) if len(ps) > 0 else None)
            valores_2.append(ps[1].get_text(strip=True) if len(ps) > 1 else None)
        dados_lutador_1.append(valores_1)
        dados_lutador_2.append(valores_2)
    
    return headers, dados_lutador_1, dados_lutador_2

In [ ]:
tabelas_html = soup.find_all("table")
print(len(tabelas_html))  # deve dar 4, igual o pd.read_html

headers, l1, l2 = extrair_tabela_por_lutador(tabelas_html[0])
print(headers)
print(l1)
print(l2)

4
['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
[['Joshua Van', '1', '181 of 305', '59%', '259 of 392', '0 of 0', '---', '1', '1', '2:13']]
[['Alexandre Pantoja', '0', '147 of 284', '51%', '159 of 300', '5 of 18', '27%', '0', '0', '9:11']]


In [ ]:
df_totals = pd.DataFrame([l1[0], l2[0]], columns=headers)
df_totals

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl
0,Joshua Van,1,181 of 305,59%,259 of 392,0 of 0,---,1,1,2:13
1,Alexandre Pantoja,0,147 of 284,51%,159 of 300,5 of 18,27%,0,0,9:11


In [ ]:
headers2, l1_2, l2_2 = extrair_tabela_por_lutador(tabelas_html[1])
print(headers2)
print(l1_2)
print(l2_2)

['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td %', 'Td %', 'Sub. att', 'Rev.', 'Ctrl', 'Round 1', 'Round 2', 'Round 3', 'Round 4', 'Round 5']
[[], ['Joshua Van', '0', '13 of 27', '48%', '46 of 61', '0 of 0', '---', '0', '1', '0:45'], [], ['Joshua Van', '0', '37 of 64', '57%', '48 of 76', '0 of 0', '---', '0', '0', '0:20'], [], ['Joshua Van', '0', '65 of 98', '66%', '67 of 101', '0 of 0', '---', '0', '0', '0:14'], [], ['Joshua Van', '0', '14 of 28', '50%', '41 of 57', '0 of 0', '---', '0', '0', '0:00'], [], ['Joshua Van', '1', '52 of 88', '59%', '57 of 97', '0 of 0', '---', '1', '0', '0:54']]
[[], ['Alexandre Pantoja', '0', '27 of 45', '60%', '30 of 48', '2 of 3', '66%', '0', '0', '2:25'], [], ['Alexandre Pantoja', '0', '32 of 76', '42%', '33 of 78', '1 of 5', '20%', '0', '0', '1:31'], [], ['Alexandre Pantoja', '0', '36 of 71', '50%', '37 of 72', '1 of 6', '16%', '0', '0', '1:17'], [], ['Alexandre Pantoja', '0', '14 of 22', '63%', '21 of 32', '1 of 2', '50%', '0', '0', 

In [ ]:
def extrair_tabela_por_lutador(tabela_html):
    linhas = tabela_html.find_all("tr")
    
    # cabeçalhos: só da primeira linha (<tr>)
    headers = [th.get_text(strip=True) for th in linhas[0].find_all("th")]
    
    dados_lutador_1 = []
    dados_lutador_2 = []
    
    for linha in linhas[1:]:
        celulas = linha.find_all("td", class_="b-fight-details__table-col")
        if not celulas:
            continue  # pula linhas vazias/estruturais
        valores_1 = []
        valores_2 = []
        for celula in celulas:
            ps = celula.find_all("p", class_="b-fight-details__table-text")
            valores_1.append(ps[0].get_text(strip=True) if len(ps) > 0 else None)
            valores_2.append(ps[1].get_text(strip=True) if len(ps) > 1 else None)
        dados_lutador_1.append(valores_1)
        dados_lutador_2.append(valores_2)
    
    return headers, dados_lutador_1, dados_lutador_2

In [ ]:
headers2, l1_2, l2_2 = extrair_tabela_por_lutador(tabelas_html[1])
print(headers2)
print(l1_2)
print(l2_2)

['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td %', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
[['Joshua Van', '0', '13 of 27', '48%', '46 of 61', '0 of 0', '---', '0', '1', '0:45'], ['Joshua Van', '0', '37 of 64', '57%', '48 of 76', '0 of 0', '---', '0', '0', '0:20'], ['Joshua Van', '0', '65 of 98', '66%', '67 of 101', '0 of 0', '---', '0', '0', '0:14'], ['Joshua Van', '0', '14 of 28', '50%', '41 of 57', '0 of 0', '---', '0', '0', '0:00'], ['Joshua Van', '1', '52 of 88', '59%', '57 of 97', '0 of 0', '---', '1', '0', '0:54']]
[['Alexandre Pantoja', '0', '27 of 45', '60%', '30 of 48', '2 of 3', '66%', '0', '0', '2:25'], ['Alexandre Pantoja', '0', '32 of 76', '42%', '33 of 78', '1 of 5', '20%', '0', '0', '1:31'], ['Alexandre Pantoja', '0', '36 of 71', '50%', '37 of 72', '1 of 6', '16%', '0', '0', '1:17'], ['Alexandre Pantoja', '0', '14 of 22', '63%', '21 of 32', '1 of 2', '50%', '0', '0', '3:58'], ['Alexandre Pantoja', '0', '38 of 70', '54%', '38 of 70', '0 of 2', '0%', '0', '0'

In [ ]:
headers_fixos = ['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']

df_por_round_l1 = pd.DataFrame(l1_2, columns=headers_fixos)
df_por_round_l1["Round"] = range(1, len(df_por_round_l1) + 1)

df_por_round_l2 = pd.DataFrame(l2_2, columns=headers_fixos)
df_por_round_l2["Round"] = range(1, len(df_por_round_l2) + 1)

df_por_round_l1

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Round
0,Joshua Van,0,13 of 27,48%,46 of 61,0 of 0,---,0,1,0:45,1
1,Joshua Van,0,37 of 64,57%,48 of 76,0 of 0,---,0,0,0:20,2
2,Joshua Van,0,65 of 98,66%,67 of 101,0 of 0,---,0,0,0:14,3
3,Joshua Van,0,14 of 28,50%,41 of 57,0 of 0,---,0,0,0:00,4
4,Joshua Van,1,52 of 88,59%,57 of 97,0 of 0,---,1,0,0:54,5


In [ ]:
df_por_round_l1["Fighter"] = "Joshua Van"  # já vem null/errado da tabela, então sobrescrevemos
df_por_round_l1["Lutador_num"] = 1

df_por_round_l2["Fighter"] = "Alexandre Pantoja"
df_por_round_l2["Lutador_num"] = 2

In [ ]:
df_por_round_l1["Fighter"] = "Joshua Van"
df_por_round_l1["Lutador_num"] = 1

df_por_round_l2["Fighter"] = "Alexandre Pantoja"
df_por_round_l2["Lutador_num"] = 2

df_por_round_l1  # <- essa linha final faz o Jupyter exibir o resultado

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Round,Lutador_num
0,Joshua Van,0,13 of 27,48%,46 of 61,0 of 0,---,0,1,0:45,1,1
1,Joshua Van,0,37 of 64,57%,48 of 76,0 of 0,---,0,0,0:20,2,1
2,Joshua Van,0,65 of 98,66%,67 of 101,0 of 0,---,0,0,0:14,3,1
3,Joshua Van,0,14 of 28,50%,41 of 57,0 of 0,---,0,0,0:00,4,1
4,Joshua Van,1,52 of 88,59%,57 of 97,0 of 0,---,1,0,0:54,5,1


In [ ]:
headers3, l1_3, l2_3 = extrair_tabela_por_lutador(tabelas_html[2])
print(headers3)
print(l1_3)
print(l2_3)

['Fighter', 'Sig. str', 'Sig. str. %', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']
[['Joshua Van', '181 of 305', '59%', '158 of 278', '17 of 20', '6 of 7', '160 of 278', '17 of 21', '4 of 6']]
[['Alexandre Pantoja', '147 of 284', '51%', '91 of 213', '30 of 37', '26 of 34', '124 of 258', '16 of 18', '7 of 8']]


In [ ]:
headers4, l1_4, l2_4 = extrair_tabela_por_lutador(tabelas_html[3])
print(headers4)
print(l1_4)
print(l2_4)

['Fighter', 'Sig. str', 'Sig. str. %', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']
[['Joshua Van', '13 of 27', '48%', '13 of 26', '0 of 1', '0 of 0', '9 of 22', '4 of 5', '0 of 0'], ['Joshua Van', '37 of 64', '57%', '27 of 52', '7 of 8', '3 of 4', '31 of 55', '6 of 9', '0 of 0'], ['Joshua Van', '65 of 98', '66%', '54 of 86', '9 of 10', '2 of 2', '58 of 91', '7 of 7', '0 of 0'], ['Joshua Van', '14 of 28', '50%', '13 of 27', '1 of 1', '0 of 0', '14 of 28', '0 of 0', '0 of 0'], ['Joshua Van', '52 of 88', '59%', '51 of 87', '0 of 0', '1 of 1', '48 of 82', '0 of 0', '4 of 6']]
[['Alexandre Pantoja', '27 of 45', '60%', '20 of 38', '4 of 4', '3 of 3', '18 of 35', '4 of 4', '5 of 6'], ['Alexandre Pantoja', '32 of 76', '42%', '21 of 60', '9 of 11', '2 of 5', '23 of 66', '8 of 9', '1 of 1'], ['Alexandre Pantoja', '36 of 71', '50%', '25 of 59', '7 of 8', '4 of 4', '33 of 67', '3 of 4', '0 of 0'], ['Alexandre Pantoja', '14 of 22', '63%', '11 of 19', '1 of 1', '2 of 2', '13 of 21', '0 of

In [ ]:
nome_1, nome_2 = "Joshua Van", "Alexandre Pantoja"

headers_totais = ['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
headers_sig = ['Fighter_sig', 'Sig. str_sig', 'Sig. str. %_sig', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']

df_resumo_l1 = pd.DataFrame([l1[0]], columns=headers_totais)
df_resumo_l1 = pd.concat([df_resumo_l1, pd.DataFrame([l1_3[0]], columns=headers_sig)], axis=1)
df_resumo_l1["Fighter"] = nome_1
df_resumo_l1 = df_resumo_l1.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])

df_resumo_l2 = pd.DataFrame([l2[0]], columns=headers_totais)
df_resumo_l2 = pd.concat([df_resumo_l2, pd.DataFrame([l2_3[0]], columns=headers_sig)], axis=1)
df_resumo_l2["Fighter"] = nome_2
df_resumo_l2 = df_resumo_l2.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])

df_resumo_luta = pd.concat([df_resumo_l1, df_resumo_l2], ignore_index=True)
df_resumo_luta

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Head,Body,Leg,Distance,Clinch,Ground
0,Joshua Van,1,181 of 305,59%,259 of 392,0 of 0,---,1,1,2:13,158 of 278,17 of 20,6 of 7,160 of 278,17 of 21,4 of 6
1,Alexandre Pantoja,0,147 of 284,51%,159 of 300,5 of 18,27%,0,0,9:11,91 of 213,30 of 37,26 of 34,124 of 258,16 of 18,7 of 8


In [ ]:
headers_totais_round = ['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
headers_sig_round = ['Fighter_sig', 'Sig. str_sig', 'Sig. str. %_sig', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']

# Lutador 1
df_round_l1 = pd.DataFrame(l1_2, columns=headers_totais_round)
df_round_l1 = pd.concat([df_round_l1, pd.DataFrame(l1_4, columns=headers_sig_round)], axis=1)
df_round_l1["Fighter"] = nome_1
df_round_l1["Round"] = range(1, len(df_round_l1) + 1)
df_round_l1 = df_round_l1.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])

# Lutador 2
df_round_l2 = pd.DataFrame(l2_2, columns=headers_totais_round)
df_round_l2 = pd.concat([df_round_l2, pd.DataFrame(l2_4, columns=headers_sig_round)], axis=1)
df_round_l2["Fighter"] = nome_2
df_round_l2["Round"] = range(1, len(df_round_l2) + 1)
df_round_l2 = df_round_l2.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])

df_por_round = pd.concat([df_round_l1, df_round_l2], ignore_index=True)
df_por_round

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Head,Body,Leg,Distance,Clinch,Ground,Round
0,Joshua Van,0,13 of 27,48%,46 of 61,0 of 0,---,0,1,0:45,13 of 26,0 of 1,0 of 0,9 of 22,4 of 5,0 of 0,1
1,Joshua Van,0,37 of 64,57%,48 of 76,0 of 0,---,0,0,0:20,27 of 52,7 of 8,3 of 4,31 of 55,6 of 9,0 of 0,2
2,Joshua Van,0,65 of 98,66%,67 of 101,0 of 0,---,0,0,0:14,54 of 86,9 of 10,2 of 2,58 of 91,7 of 7,0 of 0,3
3,Joshua Van,0,14 of 28,50%,41 of 57,0 of 0,---,0,0,0:00,13 of 27,1 of 1,0 of 0,14 of 28,0 of 0,0 of 0,4
4,Joshua Van,1,52 of 88,59%,57 of 97,0 of 0,---,1,0,0:54,51 of 87,0 of 0,1 of 1,48 of 82,0 of 0,4 of 6,5
5,Alexandre Pantoja,0,27 of 45,60%,30 of 48,2 of 3,66%,0,0,2:25,20 of 38,4 of 4,3 of 3,18 of 35,4 of 4,5 of 6,1
6,Alexandre Pantoja,0,32 of 76,42%,33 of 78,1 of 5,20%,0,0,1:31,21 of 60,9 of 11,2 of 5,23 of 66,8 of 9,1 of 1,2
7,Alexandre Pantoja,0,36 of 71,50%,37 of 72,1 of 6,16%,0,0,1:17,25 of 59,7 of 8,4 of 4,33 of 67,3 of 4,0 of 0,3
8,Alexandre Pantoja,0,14 of 22,63%,21 of 32,1 of 2,50%,0,0,3:58,11 of 19,1 of 1,2 of 2,13 of 21,0 of 0,1 of 1,4
9,Alexandre Pantoja,0,38 of 70,54%,38 of 70,0 of 2,0%,0,0,0:00,14 of 37,9 of 13,15 of 20,37 of 69,1 of 1,0 of 0,5


In [ ]:
df_resumo_luta["Method"] = "Decision - Unanimous"  # ajuste conforme o que veio do Passo 9
df_resumo_luta["Final_Round"] = 5
df_resumo_luta["Time"] = "5:00"
df_resumo_luta["Referee"] = "Herb Dean"
df_resumo_luta["Fight_URL"] = url

df_por_round["Fight_URL"] = url

In [ ]:
df_resumo_luta.to_csv("../data/raw/resumo_luta_teste.csv", index=False)
df_por_round.to_csv("../data/raw/por_round_luta_teste.csv", index=False)

print("Salvo com sucesso!")

Salvo com sucesso!


In [ ]:
df_resumo_teste, df_round_teste = extrair_dados_luta(
    "http://ufcstats.com/fight-details/568ec6af4008355a",
    driver
)

df_resumo_teste

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,...,Leg,Distance,Clinch,Ground,Resultado,Method,Final_Round,Time,Referee,Fight_URL
0,Joshua Van,1,181 of 305,59%,259 of 392,0 of 0,---,1,1,2:13,...,6 of 7,160 of 278,17 of 21,4 of 6,W,Decision - Unanimous,5,5:00,Herb Dean,http://ufcstats.com/fight-details/568ec6af4008...
1,Alexandre Pantoja,0,147 of 284,51%,159 of 300,5 of 18,27%,0,0,9:11,...,26 of 34,124 of 258,16 of 18,7 of 8,L,Decision - Unanimous,5,5:00,Herb Dean,http://ufcstats.com/fight-details/568ec6af4008...


In [ ]:
df_round_teste

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Head,Body,Leg,Distance,Clinch,Ground,Round,Fight_URL
0,Joshua Van,0,13 of 27,48%,46 of 61,0 of 0,---,0,1,0:45,13 of 26,0 of 1,0 of 0,9 of 22,4 of 5,0 of 0,1,http://ufcstats.com/fight-details/568ec6af4008...
1,Joshua Van,0,37 of 64,57%,48 of 76,0 of 0,---,0,0,0:20,27 of 52,7 of 8,3 of 4,31 of 55,6 of 9,0 of 0,2,http://ufcstats.com/fight-details/568ec6af4008...
2,Joshua Van,0,65 of 98,66%,67 of 101,0 of 0,---,0,0,0:14,54 of 86,9 of 10,2 of 2,58 of 91,7 of 7,0 of 0,3,http://ufcstats.com/fight-details/568ec6af4008...
3,Joshua Van,0,14 of 28,50%,41 of 57,0 of 0,---,0,0,0:00,13 of 27,1 of 1,0 of 0,14 of 28,0 of 0,0 of 0,4,http://ufcstats.com/fight-details/568ec6af4008...
4,Joshua Van,1,52 of 88,59%,57 of 97,0 of 0,---,1,0,0:54,51 of 87,0 of 0,1 of 1,48 of 82,0 of 0,4 of 6,5,http://ufcstats.com/fight-details/568ec6af4008...
5,Alexandre Pantoja,0,27 of 45,60%,30 of 48,2 of 3,66%,0,0,2:25,20 of 38,4 of 4,3 of 3,18 of 35,4 of 4,5 of 6,1,http://ufcstats.com/fight-details/568ec6af4008...
6,Alexandre Pantoja,0,32 of 76,42%,33 of 78,1 of 5,20%,0,0,1:31,21 of 60,9 of 11,2 of 5,23 of 66,8 of 9,1 of 1,2,http://ufcstats.com/fight-details/568ec6af4008...
7,Alexandre Pantoja,0,36 of 71,50%,37 of 72,1 of 6,16%,0,0,1:17,25 of 59,7 of 8,4 of 4,33 of 67,3 of 4,0 of 0,3,http://ufcstats.com/fight-details/568ec6af4008...
8,Alexandre Pantoja,0,14 of 22,63%,21 of 32,1 of 2,50%,0,0,3:58,11 of 19,1 of 1,2 of 2,13 of 21,0 of 0,1 of 1,4,http://ufcstats.com/fight-details/568ec6af4008...
9,Alexandre Pantoja,0,38 of 70,54%,38 of 70,0 of 2,0%,0,0,0:00,14 of 37,9 of 13,15 of 20,37 of 69,1 of 1,0 of 0,5,http://ufcstats.com/fight-details/568ec6af4008...


In [ ]:
df_resumo_teste2, df_round_teste2 = extrair_dados_luta(
    "http://ufcstats.com/fight-details/034c04e480d87b8f",
    driver
)

df_resumo_teste2

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,...,Leg,Distance,Clinch,Ground,Resultado,Method,Final_Round,Time,Referee,Fight_URL
0,Jean Silva,1,39 of 77,50%,40 of 78,1 of 2,50%,1,0,1:37,...,2 of 2,32 of 66,3 of 5,4 of 6,W,Submission,3,2:57,Dan Miragliotta,http://ufcstats.com/fight-details/034c04e480d8...
1,Jose Delgado,0,40 of 108,37%,41 of 109,0 of 1,0%,0,0,0:21,...,18 of 22,35 of 98,5 of 9,0 of 1,L,Submission,3,2:57,Dan Miragliotta,http://ufcstats.com/fight-details/034c04e480d8...


In [ ]:
df_round_teste2

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Head,Body,Leg,Distance,Clinch,Ground,Round,Fight_URL
0,Jean Silva,0,3 of 5,60%,3 of 5,0 of 1,0%,0,0,0:11,2 of 4,1 of 1,0 of 0,3 of 5,0 of 0,0 of 0,1,http://ufcstats.com/fight-details/034c04e480d8...
1,Jean Silva,0,23 of 50,46%,24 of 51,1 of 1,100%,0,0,1:02,18 of 40,5 of 10,0 of 0,17 of 40,3 of 5,3 of 5,2,http://ufcstats.com/fight-details/034c04e480d8...
2,Jean Silva,1,13 of 22,59%,13 of 22,0 of 0,---,1,0,0:24,7 of 16,4 of 4,2 of 2,12 of 21,0 of 0,1 of 1,3,http://ufcstats.com/fight-details/034c04e480d8...
3,Jose Delgado,0,17 of 34,50%,18 of 35,0 of 0,---,0,0,0:21,4 of 16,2 of 4,11 of 14,16 of 32,1 of 2,0 of 0,1,http://ufcstats.com/fight-details/034c04e480d8...
4,Jose Delgado,0,14 of 48,29%,14 of 48,0 of 0,---,0,0,0:00,8 of 42,3 of 3,3 of 3,10 of 41,4 of 7,0 of 0,2,http://ufcstats.com/fight-details/034c04e480d8...
5,Jose Delgado,0,9 of 26,34%,9 of 26,0 of 1,0%,0,0,0:00,5 of 21,0 of 0,4 of 5,9 of 25,0 of 0,0 of 1,3,http://ufcstats.com/fight-details/034c04e480d8...


In [ ]:
url_evento = "http://ufcstats.com/event-details/b96619b3acd7d9da"
driver.get(url_evento)
time.sleep(5)

html_evento = driver.page_source
soup_evento = BeautifulSoup(html_evento, "html.parser")

linhas_luta = soup_evento.find_all("tr", class_="b-fight-details__table-row")
urls_lutas = [linha["data-link"] for linha in linhas_luta if linha.has_attr("data-link")]

print(len(urls_lutas))
print(urls_lutas)

12
['http://ufcstats.com/fight-details/365fd759c03e93cf', 'http://ufcstats.com/fight-details/4700d42c9869674a', 'http://ufcstats.com/fight-details/d4bd456d70965734', 'http://ufcstats.com/fight-details/53902dcb3e213b18', 'http://ufcstats.com/fight-details/8e482cead0c7eb15', 'http://ufcstats.com/fight-details/6e495deac1f57169', 'http://ufcstats.com/fight-details/e7c11d1b22d50ba0', 'http://ufcstats.com/fight-details/9c5da658c76da7c0', 'http://ufcstats.com/fight-details/e79e4d13546d2360', 'http://ufcstats.com/fight-details/360b13f97e7ca703', 'http://ufcstats.com/fight-details/e7077d030d5fc15f', 'http://ufcstats.com/fight-details/d5ebdf462c9b59b3']


In [ ]:
todos_resumos = []
todos_rounds = []

for i, url_luta in enumerate(urls_lutas):
    print(f"Processando luta {i+1}/{len(urls_lutas)}: {url_luta}")
    try:
        df_resumo, df_round = extrair_dados_luta(url_luta, driver)
        todos_resumos.append(df_resumo)
        todos_rounds.append(df_round)
    except Exception as e:
        print(f"  Erro nessa luta: {e}")

df_evento_resumo = pd.concat(todos_resumos, ignore_index=True)
df_evento_round = pd.concat(todos_rounds, ignore_index=True)

print("Concluído!")
print(df_evento_resumo.shape, df_evento_round.shape)

Processando luta 1/12: http://ufcstats.com/fight-details/365fd759c03e93cf
Processando luta 2/12: http://ufcstats.com/fight-details/4700d42c9869674a
Processando luta 3/12: http://ufcstats.com/fight-details/d4bd456d70965734
Processando luta 4/12: http://ufcstats.com/fight-details/53902dcb3e213b18
Processando luta 5/12: http://ufcstats.com/fight-details/8e482cead0c7eb15
Processando luta 6/12: http://ufcstats.com/fight-details/6e495deac1f57169
Processando luta 7/12: http://ufcstats.com/fight-details/e7c11d1b22d50ba0
Processando luta 8/12: http://ufcstats.com/fight-details/9c5da658c76da7c0
Processando luta 9/12: http://ufcstats.com/fight-details/e79e4d13546d2360
Processando luta 10/12: http://ufcstats.com/fight-details/360b13f97e7ca703
Processando luta 11/12: http://ufcstats.com/fight-details/e7077d030d5fc15f
Processando luta 12/12: http://ufcstats.com/fight-details/d5ebdf462c9b59b3
Concluído!
(24, 22) (66, 18)


In [ ]:
df_evento_resumo["Event_URL"] = url_evento
df_evento_round["Event_URL"] = url_evento

df_evento_resumo.to_csv("../data/raw/evento_teste_resumo.csv", index=False)
df_evento_round.to_csv("../data/raw/evento_teste_round.csv", index=False)

print("Evento salvo com sucesso!")

Evento salvo com sucesso!


In [ ]:
url_evento2 = "http://ufcstats.com/event-details/2144954270be834d"
df_evento2_resumo, df_evento2_round = extrair_evento(url_evento2, driver)

print(df_evento2_resumo.shape if df_evento2_resumo is not None else "Falhou")
print(df_evento2_round.shape if df_evento2_round is not None else "Falhou")

  Luta 1/14: http://ufcstats.com/fight-details/bba8c0ab999b9e6f
  Luta 2/14: http://ufcstats.com/fight-details/55f698e5c4ed2c47
  Luta 3/14: http://ufcstats.com/fight-details/c23197c7e4f2fd9d
  Luta 4/14: http://ufcstats.com/fight-details/8300a095a4908ab1
  Luta 5/14: http://ufcstats.com/fight-details/cbf9063d23513661
  Luta 6/14: http://ufcstats.com/fight-details/b90b099c1f47bb93
  Luta 7/14: http://ufcstats.com/fight-details/ad17c34e0cc7d702
  Luta 8/14: http://ufcstats.com/fight-details/b1c6ea7d7d413bd5
  Luta 9/14: http://ufcstats.com/fight-details/94d411112ebe8fad
  Luta 10/14: http://ufcstats.com/fight-details/ac3af13ffe39b1d8
  Luta 11/14: http://ufcstats.com/fight-details/1f973e21581b082a
  Luta 12/14: http://ufcstats.com/fight-details/809ebb0cfdd01ce2
  Luta 13/14: http://ufcstats.com/fight-details/e5fd7ef7ccf955b8
  Luta 14/14: http://ufcstats.com/fight-details/9226a13d001b49ec
(28, 23)
(62, 19)


In [ ]:
url_lista_eventos = "http://ufcstats.com/statistics/events/completed"
driver.get(url_lista_eventos)
time.sleep(5)

html_lista = driver.page_source
soup_lista = BeautifulSoup(html_lista, "html.parser")

links_evento = soup_lista.find_all("a", class_="b-link b-link_style_black")

eventos = []
for link in links_evento:
    nome = link.text.strip()
    url = link["href"]
    eventos.append({"nome": nome, "url": url})

print(len(eventos))
print(eventos[:5])  # mostra os 5 primeiros, pra conferir

24
[{'nome': 'UFC 331: Van vs. Pantoja 2', 'url': 'http://ufcstats.com/event-details/8a0a35e7c74bebcc'}, {'nome': 'Noche UFC: Silva vs. Delgado', 'url': 'http://ufcstats.com/event-details/638cfec7ec559d6e'}, {'nome': 'UFC Fight Night: Hooker vs. Parnasse', 'url': 'http://ufcstats.com/event-details/2144954270be834d'}, {'nome': 'UFC Fight Night: Nurmagomedov vs. Song', 'url': 'http://ufcstats.com/event-details/9d61d8cb1c354867'}, {'nome': 'UFC Fight Night: Hernandez vs. Rodrigues', 'url': 'http://ufcstats.com/event-details/a0a69dc9914ef6e1'}]


In [ ]:
#df_dataset_resumo, df_dataset_round = extrair_varios_eventos(eventos, driver, limite=3)

#print(df_dataset_resumo.shape)
#print(df_dataset_round.shape)

In [2]:
# Já executado: o resultado do scraping acima foi salvo nestes arquivos (lidos na célula seguinte).
#df_dataset_resumo.to_csv("../data/raw/dataset_3eventos_resumo.csv", index=False)
#df_dataset_round.to_csv("../data/raw/dataset_3eventos_round.csv", index=False)

#print("Dataset de 3 eventos salvo com sucesso!")

In [3]:
df_dataset_resumo = pd.read_csv("../data/raw/dataset_3eventos_resumo.csv")
df_dataset_round = pd.read_csv("../data/raw/dataset_3eventos_round.csv")

In [4]:
df_dataset_resumo.dtypes

Fighter          str
KD             int64
Sig. str.        str
Sig. str. %      str
Total str.       str
Td               str
Td %             str
Sub. att       int64
Rev.           int64
Ctrl             str
Head             str
Body             str
Leg              str
Distance         str
Clinch           str
Ground           str
Resultado        str
Method           str
Final_Round    int64
Time             str
Referee          str
Fight_URL        str
Event_URL        str
Event_Name       str
dtype: object

In [5]:
print(df_dataset_resumo["Sig. str."].unique()[:5])
print(df_dataset_resumo["Sig. str. %"].unique()[:5])
print(df_dataset_resumo["Td %"].unique()[:10])
print(df_dataset_resumo["Ctrl"].unique()[:5])

<StringArray>
['181 of 305', '147 of 284', '8 of 19', '4 of 10', '38 of 75']
Length: 5, dtype: str
<StringArray>
['59%', '51%', '42%', '40%', '50%']
Length: 5, dtype: str
<StringArray>
['---', '27%', '10%', '20%', '33%', '100%', '0%', '16%', '25%', '50%']
Length: 10, dtype: str
<StringArray>
['2:13', '9:11', '3:00', '0:06', '0:05']
Length: 5, dtype: str


In [6]:
colunas_of = ["Sig. str.", "Total str.", "Td", "Head", "Body", "Leg", "Distance", "Clinch", "Ground"]
colunas_pct = ["Sig. str. %", "Td %"]

df_resumo_limpo = df_dataset_resumo.copy()

for col in colunas_of:
    df_resumo_limpo = separar_landed_attempted(df_resumo_limpo, col)

for col in colunas_pct:
    df_resumo_limpo = limpar_percentual(df_resumo_limpo, col)

df_resumo_limpo = tempo_para_segundos(df_resumo_limpo, "Ctrl")

df_resumo_limpo.dtypes

Fighter                    str
KD                       int64
Sig. str. %              int64
Td %                   float64
Sub. att                 int64
Rev.                     int64
Resultado                  str
Method                     str
Final_Round              int64
Time                       str
Referee                    str
Fight_URL                  str
Event_URL                  str
Event_Name                 str
Sig_str_landed           int64
Sig_str_attempted        int64
Total_str_landed         int64
Total_str_attempted      int64
Td_landed                int64
Td_attempted             int64
Head_landed              int64
Head_attempted           int64
Body_landed              int64
Body_attempted           int64
Leg_landed               int64
Leg_attempted            int64
Distance_landed          int64
Distance_attempted       int64
Clinch_landed            int64
Clinch_attempted         int64
Ground_landed            int64
Ground_attempted         int64
Ctrl_sec

In [7]:
df_round_limpo = df_dataset_round.copy()

for col in colunas_of:
    df_round_limpo = separar_landed_attempted(df_round_limpo, col)

for col in colunas_pct:
    df_round_limpo = limpar_percentual(df_round_limpo, col)

df_round_limpo = tempo_para_segundos(df_round_limpo, "Ctrl")

df_round_limpo.dtypes

Fighter                    str
KD                       int64
Sig. str. %              int64
Td %                   float64
Sub. att                 int64
Rev.                     int64
Round                    int64
Fight_URL                  str
Event_URL                  str
Event_Name                 str
Sig_str_landed           int64
Sig_str_attempted        int64
Total_str_landed         int64
Total_str_attempted      int64
Td_landed                int64
Td_attempted             int64
Head_landed              int64
Head_attempted           int64
Body_landed              int64
Body_attempted           int64
Leg_landed               int64
Leg_attempted            int64
Distance_landed          int64
Distance_attempted       int64
Clinch_landed            int64
Clinch_attempted         int64
Ground_landed            int64
Ground_attempted         int64
Ctrl_seconds             int64
dtype: object

In [8]:
df_resumo_limpo.to_csv("../data/processed/dataset_3eventos_resumo_limpo.csv", index=False)
df_round_limpo.to_csv("../data/processed/dataset_3eventos_round_limpo.csv", index=False)

print("Dados processados salvos com sucesso!")

Dados processados salvos com sucesso!


In [9]:
df_resumo_limpo.describe()

,KD,Sig. str. %,Td %,Sub. att,Rev.,Final_Round,Sig_str_landed,Sig_str_attempted,Total_str_landed,Total_str_attempted,...,Body_attempted,Leg_landed,Leg_attempted,Distance_landed,Distance_attempted,Clinch_landed,Clinch_attempted,Ground_landed,Ground_attempted,Ctrl_seconds
count,78.000000,78.000000,41.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,...,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000
mean,0.294872,45.602564,37.243902,0.282051,0.051282,2.358974,42.564103,95.102564,57.948718,112.589744,...,11.461538,6.294872,8.102564,33.000000,82.205128,5.410256,7.384615,4.153846,5.512821,110.512821
std,0.560820,13.340048,35.539260,0.600810,0.222000,1.006142,36.544081,75.495005,46.558132,83.975327,...,10.828884,6.943387,8.553255,32.677487,72.011608,6.943998,8.980832,7.138840,9.160536,157.011766
min,0.000000,16.000000,0.000000,0.000000,0.000000,1.000000,1.000000,2.000000,1.000000,2.000000,...,0.000000,0.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,37.000000,0.000000,0.000000,0.000000,1.000000,14.250000,31.250000,20.000000,40.500000,...,3.000000,1.000000,1.250000,7.500000,24.000000,0.000000,0.250000,0.000000,0.000000,2.000000
50%,0.000000,44.500000,33.000000,0.000000,0.000000,3.000000,33.000000,76.000000,50.000000,108.000000,...,7.500000,3.000000,5.000000,21.000000,63.000000,2.500000,4.000000,1.000000,1.000000,32.000000
75%,0.750000,53.000000,54.000000,0.000000,0.000000,3.000000,66.250000,138.500000,82.250000,152.500000,...,17.000000,11.500000,12.750000,55.000000,130.500000,8.000000,10.750000,4.000000,7.500000,178.500000
max,3.000000,75.000000,100.000000,3.000000,1.000000,5.000000,181.000000,305.000000,259.000000,392.000000,...,45.000000,26.000000,34.000000,160.000000,278.000000,35.000000,39.000000,32.000000,38.000000,620.000000


In [10]:
#df_24eventos_resumo, df_24eventos_round = extrair_varios_eventos(eventos, driver, limite=24)

#print(df_24eventos_resumo.shape)
#print(df_24eventos_round.shape)

In [11]:
try:
    print(driver.current_url)
except Exception as e:
    print("Driver morreu:", e)

Driver morreu: name 'driver' is not defined


In [12]:
# Sem refazer o scraping: lê o CSV bruto salvo incrementalmente por extrair_varios_eventos.
# O arquivo hoje tem todos os eventos coletados depois; os 24 primeiros são os desta etapa.
df_24eventos_resumo = pd.read_csv("../data/raw/eventos_resumo.csv")
df_24eventos_round = pd.read_csv("../data/raw/eventos_round.csv")

urls_24_eventos = list(dict.fromkeys(df_24eventos_resumo["Event_URL"]))[:24]
df_24eventos_resumo = df_24eventos_resumo[df_24eventos_resumo["Event_URL"].isin(urls_24_eventos)]
df_24eventos_round = df_24eventos_round[df_24eventos_round["Event_URL"].isin(urls_24_eventos)]

print(df_24eventos_resumo["Event_Name"].unique())

<StringArray>
[                 'UFC 331: Van vs. Pantoja 2',
                'Noche UFC: Silva vs. Delgado',
        'UFC Fight Night: Hooker vs. Parnasse',
      'UFC Fight Night: Nurmagomedov vs. Song',
    'UFC Fight Night: Hernandez vs. Rodrigues',
        'UFC 330: Makhachev vs. Machado Garry',
        'UFC Fight Night: Gamrot vs. Salkilld',
        'UFC Fight Night: Medic vs. Rodriguez',
        'UFC Fight Night: Ankalaev vs. Guskov',
       'UFC Fight Night: Du Plessis vs. Usman',
            'UFC 329: McGregor vs. Holloway 2',
          'UFC Fight Night: Fiziev vs. Torres',
         'UFC Fight Night: Kape vs. Horiguchi',
                             'UFC Freedom 250',
        'UFC Fight Night: Muhammad vs. Bonfim',
        'UFC Fight Night: Song vs. Figueiredo',
            'UFC Fight Night: Allen vs. Costa',
             'UFC 328: Chimaev vs. Strickland',
 'UFC Fight Night: Della Maddalena vs. Prates',
         'UFC Fight Night: Sterling vs. Zalal',
           'UFC Fight Nigh

In [13]:
def extrair_varios_eventos(eventos, driver, limite=None, arquivo_resumo="../data/raw/eventos_resumo.csv", arquivo_round="../data/raw/eventos_round.csv"):
    lista_eventos = eventos[:limite] if limite else eventos
    
    for i, evento in enumerate(lista_eventos):
        print(f"=== Evento {i+1}/{len(lista_eventos)}: {evento['nome']} ===")
        try:
            df_resumo, df_round = extrair_evento(evento["url"], driver)
            if df_resumo is not None:
                df_resumo["Event_Name"] = evento["nome"]
                df_round["Event_Name"] = evento["nome"]
                
                # Salva incrementalmente: cria o arquivo na primeira vez, depois só adiciona (append)
                escrever_cabecalho_resumo = not os.path.exists(arquivo_resumo)
                escrever_cabecalho_round = not os.path.exists(arquivo_round)
                
                df_resumo.to_csv(arquivo_resumo, mode="a", header=escrever_cabecalho_resumo, index=False)
                df_round.to_csv(arquivo_round, mode="a", header=escrever_cabecalho_round, index=False)
                
                print(f"  Salvo com sucesso ({len(df_resumo)} linhas de resumo, {len(df_round)} de round)")
        except Exception as e:
            print(f"  Erro nesse evento: {e}")
        time.sleep(2)
    
    print("Concluído! Recarregando dataset final do disco...")
    return pd.read_csv(arquivo_resumo), pd.read_csv(arquivo_round)

In [14]:
#df_24eventos_resumo, df_24eventos_round = extrair_varios_eventos(eventos, driver, limite=24)

In [15]:
print(df_24eventos_resumo.shape)
print(df_24eventos_round.shape)
print(df_24eventos_resumo["Event_Name"].nunique())

(600, 24)
(1342, 20)
24


In [16]:
df_24_resumo_limpo = df_24eventos_resumo.copy()

for col in colunas_of:
    df_24_resumo_limpo = separar_landed_attempted(df_24_resumo_limpo, col)

for col in colunas_pct:
    df_24_resumo_limpo = limpar_percentual(df_24_resumo_limpo, col)

df_24_resumo_limpo = tempo_para_segundos(df_24_resumo_limpo, "Ctrl")

df_24_resumo_limpo.dtypes

Fighter                    str
KD                       int64
Sig. str. %            float64
Td %                   float64
Sub. att                 int64
Rev.                     int64
Resultado                  str
Method                     str
Final_Round              int64
Time                       str
Referee                    str
Fight_URL                  str
Event_URL                  str
Event_Name                 str
Sig_str_landed           int64
Sig_str_attempted        int64
Total_str_landed         int64
Total_str_attempted      int64
Td_landed                int64
Td_attempted             int64
Head_landed              int64
Head_attempted           int64
Body_landed              int64
Body_attempted           int64
Leg_landed               int64
Leg_attempted            int64
Distance_landed          int64
Distance_attempted       int64
Clinch_landed            int64
Clinch_attempted         int64
Ground_landed            int64
Ground_attempted         int64
Ctrl_sec

In [17]:
import os
print(os.getcwd())
print(os.path.exists("../data/raw/eventos_resumo.csv"))

C:\Users\User\Desktop\ProjetoUFC\ProjetoUFC\notebooks
True


In [18]:
df_24_round_limpo = df_24eventos_round.copy()

for col in colunas_of:
    df_24_round_limpo = separar_landed_attempted(df_24_round_limpo, col)

for col in colunas_pct:
    df_24_round_limpo = limpar_percentual(df_24_round_limpo, col)

df_24_round_limpo = tempo_para_segundos(df_24_round_limpo, "Ctrl")

df_24_round_limpo.dtypes

Fighter                    str
KD                       int64
Sig. str. %            float64
Td %                   float64
Sub. att                 int64
Rev.                     int64
Round                    int64
Fight_URL                  str
Event_URL                  str
Event_Name                 str
Sig_str_landed           int64
Sig_str_attempted        int64
Total_str_landed         int64
Total_str_attempted      int64
Td_landed                int64
Td_attempted             int64
Head_landed              int64
Head_attempted           int64
Body_landed              int64
Body_attempted           int64
Leg_landed               int64
Leg_attempted            int64
Distance_landed          int64
Distance_attempted       int64
Clinch_landed            int64
Clinch_attempted         int64
Ground_landed            int64
Ground_attempted         int64
Ctrl_seconds             int64
dtype: object

In [19]:
df_24_resumo_limpo[df_24_resumo_limpo["Sig. str. %"].isna()]

,Fighter,KD,Sig. str. %,Td %,Sub. att,Rev.,Resultado,Method,Final_Round,Time,...,Body_attempted,Leg_landed,Leg_attempted,Distance_landed,Distance_attempted,Clinch_landed,Clinch_attempted,Ground_landed,Ground_attempted,Ctrl_seconds
93,Kevin Borjas,0,NaN,NaN,0,0,L,Submission,1,4:14,...,0,0,0,0,0,0,0,0,0,0


In [20]:
df_24_resumo_limpo.describe()

,KD,Sig. str. %,Td %,Sub. att,Rev.,Final_Round,Sig_str_landed,Sig_str_attempted,Total_str_landed,Total_str_attempted,...,Body_attempted,Leg_landed,Leg_attempted,Distance_landed,Distance_attempted,Clinch_landed,Clinch_attempted,Ground_landed,Ground_attempted,Ctrl_seconds
count,600.000000,599.000000,353.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,...,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000
mean,0.298333,47.490818,34.002833,0.320000,0.106667,2.236667,39.663333,82.830000,55.658333,101.410000,...,11.320000,6.198333,7.773333,31.756667,72.010000,4.066667,5.523333,3.840000,5.296667,108.558333
std,0.565565,15.216811,34.320059,0.696473,0.349513,1.043617,36.082913,71.816319,46.274657,80.504452,...,12.284779,7.993257,9.587239,32.000534,67.460641,6.422432,8.328369,7.976451,11.699721,161.479212
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,38.000000,0.000000,0.000000,0.000000,1.000000,12.000000,27.000000,18.000000,33.000000,...,2.000000,1.000000,1.000000,8.000000,20.750000,0.000000,0.000000,0.000000,0.000000,2.000000
50%,0.000000,47.000000,33.000000,0.000000,0.000000,2.500000,30.000000,65.500000,47.000000,88.000000,...,8.000000,3.000000,5.000000,21.000000,52.000000,2.000000,2.000000,0.000000,1.000000,34.000000
75%,0.250000,56.000000,54.000000,0.000000,0.000000,3.000000,57.000000,119.000000,80.000000,146.000000,...,16.000000,9.000000,11.000000,45.000000,105.000000,6.000000,7.000000,4.000000,6.000000,146.250000
max,3.000000,100.000000,100.000000,5.000000,2.000000,5.000000,252.000000,513.000000,259.000000,517.000000,...,76.000000,61.000000,67.000000,234.000000,484.000000,60.000000,73.000000,71.000000,165.000000,829.000000


In [21]:
df_24_resumo_limpo.to_csv("../data/processed/dataset_24eventos_resumo_limpo.csv", index=False)
df_24_round_limpo.to_csv("../data/processed/dataset_24eventos_round_limpo.csv", index=False)

print("Dataset de 24 eventos processado e salvo com sucesso!")

Dataset de 24 eventos processado e salvo com sucesso!


In [22]:
#eventos_completos = extrair_eventos_multiplas_paginas(driver, num_paginas=6)
#print(len(eventos_completos))

In [23]:
# Já executado: separava os eventos ainda não coletados (depende do scraping comentado acima).
#df_ja_processado = pd.read_csv("../data/raw/eventos_resumo.csv")
#urls_ja_processadas = set(df_ja_processado["Event_URL"].unique())

#eventos_novos = [e for e in eventos_completos if e["url"] not in urls_ja_processadas]

#print(f"Total coletado: {len(eventos_completos)}")
#print(f"Já processados antes: {len(urls_ja_processadas)}")
#print(f"Novos a processar: {len(eventos_novos)}")

In [24]:
#df_resumo_final, df_round_final = extrair_varios_eventos(eventos_novos, driver, limite=None)

#print(df_resumo_final.shape)
#print(df_round_final.shape)

In [25]:
# Sem refazer o scraping: extrair_varios_eventos devolve o CSV inteiro relido do disco,
# então ler os arquivos diretamente dá o mesmo resultado (149 eventos).
df_resumo_final = pd.read_csv("../data/raw/eventos_resumo.csv")
df_round_final = pd.read_csv("../data/raw/eventos_round.csv")

print(df_resumo_final.shape)
print(df_round_final.shape)

(3674, 24)
(8910, 20)


In [26]:
print(df_resumo_final["Event_Name"].nunique())
print(df_resumo_final["Event_URL"].nunique())

149
149


In [27]:
print(df_resumo_final["Event_Name"].nunique())
print(df_resumo_final["Event_URL"].nunique())

149
149


In [28]:
df_final_resumo_limpo = df_resumo_final.copy()

for col in colunas_of:
    df_final_resumo_limpo = separar_landed_attempted(df_final_resumo_limpo, col)

for col in colunas_pct:
    df_final_resumo_limpo = limpar_percentual(df_final_resumo_limpo, col)

df_final_resumo_limpo = tempo_para_segundos(df_final_resumo_limpo, "Ctrl")

df_final_resumo_limpo.dtypes

Fighter                    str
KD                       int64
Sig. str. %            float64
Td %                   float64
Sub. att                 int64
Rev.                     int64
Resultado                  str
Method                     str
Final_Round              int64
Time                       str
Referee                    str
Fight_URL                  str
Event_URL                  str
Event_Name                 str
Sig_str_landed           int64
Sig_str_attempted        int64
Total_str_landed         int64
Total_str_attempted      int64
Td_landed                int64
Td_attempted             int64
Head_landed              int64
Head_attempted           int64
Body_landed              int64
Body_attempted           int64
Leg_landed               int64
Leg_attempted            int64
Distance_landed          int64
Distance_attempted       int64
Clinch_landed            int64
Clinch_attempted         int64
Ground_landed            int64
Ground_attempted         int64
Ctrl_sec

In [29]:
df_final_round_limpo = df_round_final.copy()

for col in colunas_of:
    df_final_round_limpo = separar_landed_attempted(df_final_round_limpo, col)

for col in colunas_pct:
    df_final_round_limpo = limpar_percentual(df_final_round_limpo, col)

df_final_round_limpo = tempo_para_segundos(df_final_round_limpo, "Ctrl")

df_final_round_limpo.dtypes

Fighter                    str
KD                       int64
Sig. str. %            float64
Td %                   float64
Sub. att                 int64
Rev.                     int64
Round                    int64
Fight_URL                  str
Event_URL                  str
Event_Name                 str
Sig_str_landed           int64
Sig_str_attempted        int64
Total_str_landed         int64
Total_str_attempted      int64
Td_landed                int64
Td_attempted             int64
Head_landed              int64
Head_attempted           int64
Body_landed              int64
Body_attempted           int64
Leg_landed               int64
Leg_attempted            int64
Distance_landed          int64
Distance_attempted       int64
Clinch_landed            int64
Clinch_attempted         int64
Ground_landed            int64
Ground_attempted         int64
Ctrl_seconds             int64
dtype: object

In [30]:
df_final_resumo_limpo.describe()

,KD,Sig. str. %,Td %,Sub. att,Rev.,Final_Round,Sig_str_landed,Sig_str_attempted,Total_str_landed,Total_str_attempted,...,Body_attempted,Leg_landed,Leg_attempted,Distance_landed,Distance_attempted,Clinch_landed,Clinch_attempted,Ground_landed,Ground_attempted,Ctrl_seconds
count,3674.000000,3669.000000,2339.000000,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000,...,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000,3674.000000
mean,0.222101,47.821750,36.390338,0.277354,0.123027,2.425150,43.676375,92.122210,61.058792,113.000272,...,13.224551,7.128198,8.843767,35.450191,80.690800,4.171475,5.759118,4.054709,5.672292,123.475776
std,0.490244,14.687247,34.982842,0.642529,0.400922,1.030691,35.708527,73.908008,46.654214,81.680583,...,13.274416,8.402815,10.168340,32.748592,71.257884,6.397828,8.255810,8.745268,12.023136,170.508840
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,39.000000,0.000000,0.000000,0.000000,1.000000,16.000000,33.000000,24.000000,44.000000,...,3.000000,1.000000,2.000000,10.000000,24.000000,0.000000,0.000000,0.000000,0.000000,4.000000
50%,0.000000,48.000000,33.000000,0.000000,0.000000,3.000000,36.000000,75.000000,54.000000,102.000000,...,9.000000,4.000000,5.500000,26.000000,61.000000,2.000000,3.000000,0.000000,1.000000,48.000000
75%,0.000000,56.000000,60.000000,0.000000,0.000000,3.000000,64.000000,135.000000,88.000000,163.000000,...,19.000000,10.000000,13.000000,52.000000,121.000000,5.000000,8.000000,4.000000,6.000000,180.750000
max,6.000000,100.000000,100.000000,6.000000,5.000000,5.000000,252.000000,513.000000,529.000000,567.000000,...,127.000000,95.000000,102.000000,234.000000,504.000000,66.000000,86.000000,136.000000,165.000000,1300.000000


In [31]:
df_final_resumo_limpo.to_csv("../data/processed/dataset_final_resumo_limpo.csv", index=False)
df_final_round_limpo.to_csv("../data/processed/dataset_final_round_limpo.csv", index=False)

print("Dataset final (149 eventos) processado e salvo com sucesso!")

Dataset final (149 eventos) processado e salvo com sucesso!


In [32]:
print(criar_driver)
print(extrair_dados_luta)
print(colunas_of)
print(colunas_pct)

<function criar_driver at 0x0000024A4461B1C0>
<function extrair_dados_luta at 0x0000024A64066770>
['Sig. str.', 'Total str.', 'Td', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']
['Sig. str. %', 'Td %']


In [33]:
df_teste = pd.read_csv("../data/processed/dataset_24eventos_resumo_limpo.csv")
print(df_teste.shape)
print(df_teste.dtypes.head())

(600, 33)
Fighter            str
KD               int64
Sig. str. %    float64
Td %           float64
Sub. att         int64
dtype: object


In [34]:
df_bruto_teste = pd.read_csv("../data/raw/eventos_resumo.csv")

df_bruto_teste_copia = df_bruto_teste.copy()
for col in colunas_of:
    df_bruto_teste_copia = separar_landed_attempted(df_bruto_teste_copia, col)

print(df_bruto_teste_copia.dtypes)

Fighter                  str
KD                     int64
Sig. str. %              str
Td %                     str
Sub. att               int64
Rev.                   int64
Ctrl                     str
Resultado                str
Method                   str
Final_Round            int64
Time                     str
Referee                  str
Fight_URL                str
Event_URL                str
Event_Name               str
Sig_str_landed         int64
Sig_str_attempted      int64
Total_str_landed       int64
Total_str_attempted    int64
Td_landed              int64
Td_attempted           int64
Head_landed            int64
Head_attempted         int64
Body_landed            int64
Body_attempted         int64
Leg_landed             int64
Leg_attempted          int64
Distance_landed        int64
Distance_attempted     int64
Clinch_landed          int64
Clinch_attempted       int64
Ground_landed          int64
Ground_attempted       int64
dtype: object
